# 06_LightCurvesIsolatedStars: light curves of the isolated stars found in stage 1, and their behaviour over the LSSTCam focal plane

The stage-1 task `associateIsolatedStar` builds, per tract, the catalog of isolated stars (`isolated_star_association`) and the table of all their single-visit measurements (`isolated_star`). Each isolated star is therefore observed in many visits, on different detectors: this is a ready-made photometric-stability data set.

Questions:
1. How many isolated stars, in which bands, with how many visits each?
2. What is the repeatability of the PSF photometry as a function of magnitude, compared with the photometric errors (`rms`/MAD of a light curve versus `psfFluxErr`)?
3. What do typical stable and variable light curves look like?
4. Camera: after removing the per-visit offset (clouds, zero point), is there a residual dependence on the **detector** (focal-plane map, per raft), on the **position on the detector** (x, y), on the **airmass** or on the **time**?

Residual of a measurement: `dm = mag - median(mag of the star in that band)`. Per-visit offset: median of `dm` over all bright stars of the visit. Detector effect: median of `dm - visit_offset` over the bright stars of a detector.

Run with the `lsst_distrib` kernel at CC-IN2P3 (butler `main`). The column names of `isolated_star` depend on the release: they are printed in section 1 and chosen in `FLUX_CANDIDATES`, `PIXEL_FLAGS`.

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-07
- input collection : `u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1`
- stage-1 tasks involved : `consolidateSingleVisitStar` -> `associateIsolatedStar` (datasets `isolated_star_association`, `isolated_star`), `consolidateVisitSummary` (`preliminary_visit_summary`)
- see also : `05_CheckStage1AllOutputs.ipynb` (are all the stage-1 outputs present?)

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '06_LightCurvesIsolatedStars'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.labelsize'] = 'x-large'
plt.rcParams['axes.titlesize'] = 'x-large'
plt.rcParams['xtick.labelsize'] = 'large'
plt.rcParams['ytick.labelsize'] = 'large'


def savefig(fig, name):
    # every figure is saved as PDF and PNG
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{NB_TAG}_{name}.{ext}', bbox_inches='tight', dpi=150)

In [ ]:
REPO = 'main'
USER = os.getenv('USER')
INSTRUMENT = 'LSSTCam'

RUN_TAG = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'
STAGE1 = f'u/{USER}/LSSTCam/{RUN_TAG}/stage1'
DEFAULTS = 'LSSTCam/defaults'
TRACTS = [9813]

# column choice (checked against the real columns in section 1)
FLUX_CANDIDATES = ['psfFlux', 'psf_flux', 'psfFlux_instFlux']   # first one present is used; nJy if calibrated
PIXEL_FLAGS = ['pixelFlags_edge', 'pixelFlags_saturatedCenter', 'pixelFlags_interpolatedCenter',
               'pixelFlags_crCenter', 'pixelFlags_bad']

# selection
MIN_SNR = 20.0            # per measurement: psf flux / error
MIN_OBS = 10              # minimum number of measurements of a star in a band
MAG_BRIGHT = (17.0, 19.5) # stars used for the camera systematics (bright, unsaturated)

# light curves
BAND_LC = 'i'
N_LC = 6

STRIDE_VISITS = 1   # only used if the flux is an instFlux and zero points must be read from the visit summaries

## 1. Load the isolated stars

`isolated_star_association`: one row per isolated star (position, number of sources per band). `isolated_star`: one row per measurement (`isolated_star_id`, `visit`, `detector`, position on the detector, fluxes, flags).

In [ ]:
butler = mon.get_butler(REPO, collections=[STAGE1, DEFAULTS])
reg = butler.registry


def as_df(obj):
    return obj.to_pandas() if hasattr(obj, 'to_pandas') else obj


def refs_in_tracts(dataset_type):
    kept = []
    for ref in reg.queryDatasets(dataset_type, collections=[STAGE1], findFirst=True):
        try:
            if ref.dataId['tract'] not in TRACTS:
                continue
        except KeyError:
            pass
        kept.append(ref)
    return kept


assoc_refs = refs_in_tracts('isolated_star_association')
src_refs = refs_in_tracts('isolated_star')
print(len(assoc_refs), 'isolated_star_association dataset(s),', len(src_refs), 'isolated_star dataset(s)')

assoc = pd.concat([as_df(butler.get(ref)) for ref in assoc_refs], ignore_index=True)
srcs = pd.concat([as_df(butler.get(ref)) for ref in src_refs], ignore_index=True)
print('isolated stars:', len(assoc), ' measurements:', len(srcs))
print()
print('association columns:', list(assoc.columns))
print()
print('source columns:', list(srcs.columns))
display(srcs.head(3))

In [ ]:
FLUX_COL = next((c for c in FLUX_CANDIDATES if c in srcs.columns), None)
if FLUX_COL is None:
    raise RuntimeError('no flux column found, edit FLUX_CANDIDATES. Columns: ' + ', '.join(srcs.columns))
ERR_COL = FLUX_COL + 'Err'
if ERR_COL not in srcs.columns:
    raise RuntimeError(f'no error column {ERR_COL}')
IS_INSTFLUX = FLUX_COL.endswith('instFlux')
base = FLUX_COL[:-len('_instFlux')] if IS_INSTFLUX else FLUX_COL
FLAG_COL = base + '_flag' if base + '_flag' in srcs.columns else None
BAD_FLAGS = [c for c in PIXEL_FLAGS if c in srcs.columns]

print('flux column :', FLUX_COL, '(instrumental, zero points needed)' if IS_INSTFLUX else '(taken as calibrated, nJy)')
print('error column:', ERR_COL)
print('flag column :', FLAG_COL)
print('pixel flags :', BAD_FLAGS)
print('x, y columns present:', {'x', 'y'} <= set(srcs.columns))

## 2. Visit metadata (time, band, zenith angle) and zero points

The exposure records give the mid-exposure MJD, the physical filter (hence the band) and the zenith angle. For LSSTCam the visit id equals the exposure id. Zero points are read from `preliminary_visit_summary` only if the flux column is an instrumental flux.

In [ ]:
visits = sorted(int(v) for v in srcs['visit'].unique())
print(len(visits), 'visits with isolated-star measurements')

band_of = {rec.name: rec.band for rec in reg.queryDimensionRecords('physical_filter', instrument=INSTRUMENT)}
records = reg.queryDimensionRecords('exposure', where='instrument = inst AND exposure IN (ids)',
                                    bind={'inst': INSTRUMENT, 'ids': visits})
visit_df = pd.DataFrame([{
    'visit': rec.id,
    'mjd': 0.5 * (rec.timespan.begin.mjd + rec.timespan.end.mjd),
    'band_v': band_of.get(rec.physical_filter),
    'exp_time': getattr(rec, 'exposure_time', np.nan),
    'zenith_angle': getattr(rec, 'zenith_angle', np.nan),
} for rec in records])
visit_df['airmass'] = 1.0 / np.cos(np.deg2rad(visit_df['zenith_angle'].astype(float)))
print(visit_df.shape)
print(visit_df.band_v.value_counts().to_string())


def load_zero_points(visit_list):
    parts = []
    for v in visit_list:
        try:
            tab = butler.get('preliminary_visit_summary', visit=int(v), instrument=INSTRUMENT).asAstropy()
        except Exception:
            continue
        parts.append(pd.DataFrame(dict(visit=int(v), detector=np.asarray(tab['id']),
                                       zeroPoint=np.asarray(tab['zeroPoint']))))
    return pd.concat(parts, ignore_index=True)

## 3. Measurement table

Quality cuts: finite and positive flux, psf-flux flag and pixel flags not set, `flux/error > MIN_SNR`, at least `MIN_OBS` measurements of the star in its band. Magnitudes are AB: `mag = 31.4 - 2.5 log10(flux in nJy)`.

In [ ]:
meas = srcs.merge(visit_df, on='visit', how='left')
meas['band'] = meas['band_v']

if IS_INSTFLUX:
    zp_df = load_zero_points(visits[::STRIDE_VISITS])
    meas = meas.merge(zp_df, on=['visit', 'detector'], how='inner')
    scale = 10 ** (-0.4 * (meas['zeroPoint'].to_numpy(float) - 31.4))
else:
    scale = 1.0

flux = meas[FLUX_COL].to_numpy(float) * scale
err = meas[ERR_COL].to_numpy(float) * scale
meas['flux_njy'] = flux
meas['err_njy'] = err

good = np.isfinite(flux) & np.isfinite(err) & (flux > 0) & (err > 0)
if FLAG_COL:
    good &= ~meas[FLAG_COL].to_numpy(bool)
for col in BAD_FLAGS:
    good &= ~meas[col].to_numpy(bool)
good &= meas['band'].notna().to_numpy()
with np.errstate(divide='ignore', invalid='ignore'):
    good &= (flux / err) > MIN_SNR
print(f'{good.sum()} / {len(meas)} measurements pass the quality cuts')

meas = meas[good].copy()
meas['mag'] = 31.4 - 2.5 * np.log10(meas['flux_njy'])
meas['mag_err'] = 1.0857 * meas['err_njy'] / meas['flux_njy']

KEYS = ['isolated_star_id', 'band']
n_per = meas.groupby(KEYS)['visit'].transform('size')
meas = meas[n_per >= MIN_OBS].copy()
print(f'{len(meas)} measurements of {meas.isolated_star_id.nunique()} stars with at least {MIN_OBS} measurements in a band')
print(meas.groupby('band')['isolated_star_id'].nunique().rename('n_stars').to_string())

## 4. Repeatability of the photometry

For each (star, band): median magnitude, robust scatter (MAD x 1.4826) and `chi2/dof` of the light curve around its median, using the pipeline errors. The binned median of the scatter is compared to the binned median of the quoted errors: where they depart, the scatter is dominated by something else than photon noise (calibration, PSF model, flat field, variability).

In [ ]:
meas['mag_med'] = meas.groupby(KEYS)['mag'].transform('median')
meas['dm'] = meas['mag'] - meas['mag_med']
meas['absdm'] = meas['dm'].abs()
meas['chi2'] = (meas['dm'] / meas['mag_err']) ** 2

stars = meas.groupby(KEYS).agg(n=('mag', 'size'), mag_med=('mag_med', 'first'), rms=('dm', 'std'),
                               mad=('absdm', 'median'), err_med=('mag_err', 'median'), chi2_sum=('chi2', 'sum'))
stars['mad'] = 1.4826 * stars['mad']
stars['chi2dof'] = stars['chi2_sum'] / (stars['n'] - 1)
bands = [b for b in 'ugrizy' if b in stars.index.get_level_values('band')]
print('bands:', bands)
display(stars.groupby('band')[['n', 'mag_med', 'mad', 'err_med', 'chi2dof']].median())


def binned_median(x, y, edges):
    grouped = pd.Series(np.asarray(y)).groupby(pd.cut(pd.Series(np.asarray(x)), edges), observed=True).median()
    return np.array([iv.mid for iv in grouped.index]), grouped.to_numpy()


edges = np.arange(15.0, 26.0, 0.5)
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
for ax, band in zip(axes.flat, 'ugrizy'):
    if band not in bands:
        ax.set_visible(False)
        continue
    s = stars.xs(band, level='band')
    ax.semilogy(s['mag_med'], 1000 * s['mad'], '.', ms=2, alpha=0.3, color='tab:blue')
    xc, ym = binned_median(s['mag_med'], 1000 * s['mad'], edges)
    xc2, ye = binned_median(s['mag_med'], 1000 * s['err_med'], edges)
    ax.plot(xc, ym, 'k-', lw=2, label='median scatter (MAD)')
    ax.plot(xc2, ye, 'r--', lw=2, label='median quoted error')
    ax.set_title(f'{band} band, {len(s)} stars')
    ax.set_xlabel('median magnitude (AB)')
    ax.set_ylabel('scatter (mmag)')
    ax.set_ylim(1, 500)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Repeatability of the PSF photometry of isolated stars', fontsize=16)
savefig(fig, 'scatter_vs_mag')
plt.show()

## 5. Light curves

In band `BAND_LC`, among stars with at least 20 measurements and a magnitude in `MAG_BRIGHT`: the most stable ones (smallest scatter) and the most variable ones (largest `chi2/dof`). The most variable are either real variables or measurements affected by a systematic (bad detector, bad visit): section 6 tells which.

In [ ]:
if BAND_LC not in bands:
    BAND_LC = bands[0]
    print('BAND_LC not available, using', BAND_LC)

sb = stars.xs(BAND_LC, level='band')
sel = sb[(sb['n'] >= 20) & sb['mag_med'].between(*MAG_BRIGHT)]
print(len(sel), 'stars selected in band', BAND_LC)
stable_ids = list(sel.sort_values('mad').index[:N_LC])
variable_ids = list(sel.sort_values('chi2dof', ascending=False).index[:N_LC])


def plot_lightcurves(star_ids, title, name):
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    for ax, sid in zip(axes.flat, star_ids):
        d = meas[(meas['isolated_star_id'] == sid) & (meas['band'] == BAND_LC)].sort_values('mjd')
        ax.errorbar(d['mjd'], d['mag'], yerr=d['mag_err'], fmt='o', ms=3, lw=0.8)
        ax.invert_yaxis()
        row = sel.loc[sid]
        ax.set_title(f'star {sid}: {BAND_LC}={row.mag_med:.2f}, MAD={1000 * row.mad:.1f} mmag, chi2/dof={row.chi2dof:.1f}', fontsize=11)
        ax.set_xlabel('MJD')
        ax.set_ylabel(f'{BAND_LC} (AB)')
        ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    savefig(fig, name)
    plt.show()


plot_lightcurves(stable_ids, f'Most stable isolated stars, {BAND_LC} band', 'lightcurves_stable')
plot_lightcurves(variable_ids, f'Most variable isolated stars (largest chi2/dof), {BAND_LC} band', 'lightcurves_variable')

## 6. The camera: detector-to-detector systematics

Bright stars only (`MAG_BRIGHT`), so that the photon noise is small. `visit_offset` (median of `dm` over the whole visit) is removed first; what remains per detector is a systematic of the calibration chain (flat field / illumination correction, PSF model, preliminary photometric calibration) or of the sensor. Maps are drawn in focal-plane coordinates with the detector outlines of the camera.

In [ ]:
from lsst.afw.cameraGeom import FOCAL_PLANE
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon

try:
    camera = butler.get('camera', instrument=INSTRUMENT, exposure=visits[0])
except Exception as exc:
    print('camera from the butler failed (', type(exc).__name__, '), using lsst.obs.lsst')
    from lsst.obs.lsst import LsstCam
    camera = LsstCam().getCamera()
det_names = {det.getId(): det.getName() for det in camera}

bright = meas[meas['mag_med'].between(*MAG_BRIGHT)].copy()
bright['visit_offset'] = bright.groupby('visit')['dm'].transform('median')
bright['dmc'] = 1000.0 * (bright['dm'] - bright['visit_offset'])   # mmag
bright['dmc_det_med'] = bright.groupby('detector')['dmc'].transform('median')
bright['dmc_absdev'] = (bright['dmc'] - bright['dmc_det_med']).abs()
print(len(bright), 'bright measurements,', bright.detector.nunique(), 'detectors')

det_stats = bright.groupby('detector').agg(n=('dmc', 'size'), offset_mmag=('dmc', 'median'),
                                           mad_mmag=('dmc_absdev', 'median'))
det_stats['mad_mmag'] = 1.4826 * det_stats['mad_mmag']
det_stats['name'] = det_stats.index.map(det_names)
det_stats['raft'] = det_stats['name'].str.split('_').str[0]


def focal_plane_map(values, title, label, name, cmap='viridis', symmetric=False):
    patches, vals = [], []
    for det in camera:
        det_id = det.getId()
        if det_id not in values.index:
            continue
        corners = [(p.getX(), p.getY()) for p in det.getCorners(FOCAL_PLANE)]
        patches.append(Polygon(corners, closed=True))
        vals.append(values.loc[det_id])
    vals = np.array(vals, dtype=float)
    fig, ax = plt.subplots(figsize=(9, 8))
    pc = PatchCollection(patches, cmap=cmap, edgecolor='k', linewidth=0.3)
    pc.set_array(vals)
    if symmetric:
        lim = np.nanpercentile(np.abs(vals), 98)
        pc.set_clim(-lim, lim)
    ax.add_collection(pc)
    allc = np.concatenate([p.get_xy() for p in patches])
    ax.set_xlim(allc[:, 0].min() - 10, allc[:, 0].max() + 10)
    ax.set_ylim(allc[:, 1].min() - 10, allc[:, 1].max() + 10)
    ax.set_aspect('equal')
    ax.set_xlabel('focal plane x (mm)')
    ax.set_ylabel('focal plane y (mm)')
    ax.set_title(title)
    fig.colorbar(pc, ax=ax, label=label)
    savefig(fig, name)
    plt.show()


focal_plane_map(det_stats['offset_mmag'], 'Detector offset of bright isolated stars', 'median residual (mmag)',
                'focalplane_offset', cmap='coolwarm', symmetric=True)
focal_plane_map(det_stats['mad_mmag'], 'Detector scatter of bright isolated stars', 'robust scatter (mmag)',
                'focalplane_scatter')
focal_plane_map(det_stats['n'], 'Number of bright measurements per detector', 'N', 'focalplane_n')

raft_stats = det_stats.groupby('raft').agg(n_det=('n', 'size'), n=('n', 'sum'), offset_mmag=('offset_mmag', 'median'),
                                           scatter_mmag=('mad_mmag', 'median'))
display(raft_stats)

## 7. Position on the detector, time and airmass

- median residual in 16 x 16 bins of the pixel position (x, y), all detectors stacked: a pattern here is an illumination / pixel-area / PSF-model effect common to the detectors;
- visit offset (clouds, zero point of the preliminary calibration) versus MJD and versus airmass.

In [ ]:
if {'x', 'y'} <= set(bright.columns):
    nb = 16
    xmax, ymax = bright['x'].max(), bright['y'].max()
    bright['xbin'] = np.clip((bright['x'] / xmax * nb).astype(int), 0, nb - 1)
    bright['ybin'] = np.clip((bright['y'] / ymax * nb).astype(int), 0, nb - 1)
    grid = bright.groupby(['ybin', 'xbin'])['dmc'].median().unstack('xbin')
    grid = grid.reindex(index=range(nb), columns=range(nb))
    fig, ax = plt.subplots(figsize=(8, 7))
    lim = np.nanpercentile(np.abs(grid.to_numpy()), 98)
    im = ax.imshow(grid.to_numpy(), origin='lower', extent=[0, xmax, 0, ymax], cmap='coolwarm', vmin=-lim, vmax=lim)
    fig.colorbar(im, ax=ax, label='median residual (mmag)')
    ax.set_xlabel('x (pixel)')
    ax.set_ylabel('y (pixel)')
    ax.set_title('Residual versus position on the detector (all detectors)')
    savefig(fig, 'residual_vs_xy')
    plt.show()
else:
    print('no x, y columns in the isolated_star table: position map skipped')

visit_stats = (bright.groupby(['visit', 'band']).agg(offset_mmag=('visit_offset', 'first'), n=('dm', 'size'))
               .reset_index().merge(visit_df[['visit', 'mjd', 'airmass']], on='visit'))
visit_stats['offset_mmag'] = 1000.0 * visit_stats['offset_mmag']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for band in bands:
    v = visit_stats[visit_stats['band'] == band]
    axes[0].plot(v['mjd'], v['offset_mmag'], '.', label=band)
    axes[1].plot(v['airmass'], v['offset_mmag'], '.', label=band)
    axes[2].hist(v['offset_mmag'], bins=40, histtype='step', label=band)
axes[0].set_xlabel('MJD')
axes[1].set_xlabel('airmass')
axes[2].set_xlabel('visit offset (mmag)')
axes[0].set_ylabel('visit offset (mmag)')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('Per-visit photometric offset of bright isolated stars', fontsize=16)
savefig(fig, 'visit_offsets')
plt.show()
print(visit_stats.groupby('band')['offset_mmag'].agg(['count', 'median', 'std']))

## 8. Save the tables

In [ ]:
stars.reset_index().to_csv(DATA_DIR / 'stars_stats.csv', index=False)
det_stats.reset_index().to_csv(DATA_DIR / 'detector_stats.csv', index=False)
visit_stats.to_csv(DATA_DIR / 'visit_offsets.csv', index=False)
print('tables saved in', DATA_DIR)